# Sequence design with ProteinMPNN

#### Objective
Take 29 backbones from RFdiffusion and use ProteinMPNN to assign the amino acid residues by inverse folding, while preserving IL-2's own residues at the grafted positions. Afterward, quantify what the graft costs relative to designing the same backbones unconstrained.

#### Methodology
For each of 29 backbones there were 2 runs: fixAll with 36 fixed positions, holding the complete transplanted segments. fixContacts with 21 fixed positions, where only residues with measured β/γc burial were preserved, leaving the structurally intervening residues free to be redesigned for the new scaffold. The arms test whether freezing the whole segment buys anything over freezing only the residues that make contact. 5 sequences per backbone per arm, temperature 0.1, seed 0, Cys was excluded from the designed positions (to prevent S-S, since binding complex is in the extracellular space.) <br>
ProteinMPNN was chosen over a language model because ESM2 is sequence-only and has no inverse-folding feature. <br>
Results' interpretation scores require a reference scale -> the same protocol desibed above was applied to native Il-2 and Neo-2/15 (PDB:6DG6), whereby both are unconstrained. For the native IL-2 ther same 21 and 36 positions are frozen.

####Limitation of the design
MPNN score measures how well a sequence matches a backbone, not whether the protein folds, is stable, or binds anything.

In [ ]:
# Load the 29 Stage 2 survivors and the chain assignment from Stage 1.
# Reading survivors file rather than re-deriving the filter keeps the
# selection criterion in one place.
import os, subprocess, json, glob, re
os.environ["DGLBACKEND"] = "pytorch"

def sh(a):
    print("$", " ".join(a))
    subprocess.run(a, check=True)

sh(["pip", "-q", "install", "git+https://github.com/sokrypton/ColabDesign.git"])
sh(["pip", "-q", "install", "biopython"])

import numpy as np, pandas as pd
from google.colab import drive
drive.mount("/content/drive")

PROJ = "/content/drive/MyDrive/il2_mimetic"
OUT2 = f"{PROJ}/stage2_backbones"
OUT3 = f"{PROJ}/stage3_seqs"
os.makedirs(OUT3, exist_ok=True)

with open(f"{PROJ}/stage1_motif/params.json") as fh:
    params = json.load(fh)
CHAIN = params["chains"]

survivors = pd.read_csv(f"{OUT2}/survivors.tsv", sep="\t")
print(f"\n{len(survivors)} survivors loaded")
print(survivors.variant.value_counts().to_string())

$ pip -q install git+https://github.com/sokrypton/ColabDesign.git
$ pip -q install biopython
Mounted at /content/drive

29 survivors loaded
variant
pad0_len100    14
pad2_len100    14
pad2_len120     1


In [ ]:
# Introspect the ColabDesign MPNN API before relying on it: argument
# names and return keys are undocumented and have changed between
# releases, so they are read from the installed version rather than
# assumed
from colabdesign.mpnn import mk_mpnn_model
import inspect

mpnn = mk_mpnn_model()
for m in ("prep_inputs", "sample", "sample_parallel", "score"):
    if hasattr(mpnn, m):
        print(f"{m}{inspect.signature(getattr(mpnn, m))}\n")

print("methods:", [m for m in dir(mpnn) if not m.startswith("_")])

prep_inputs(pdb_filename=None, chain=None, homooligomer=False, ignore_missing=True, fix_pos=None, inverse=False, rm_aa=None, verbose=False, **kwargs)

sample(num=1, batch=1, temperature=0.1, rescore=False, **kwargs)

sample_parallel(batch=10, temperature=0.1, rescore=False, **kwargs)

score(seq=None, **kwargs)

methods: ['get_af_inputs', 'get_logits', 'get_unconditional_logits', 'key', 'prep_inputs', 'sample', 'sample_parallel', 'score', 'set_seed']


In [ ]:
# Single-backbone pilot. The positive control is that the sampled
# sequence reproduces IL-2's residues exactly at the fixed positions.
# Note that a cysteine survives, because rm_aa applies only to designed
# positions and C125 falls inside a frozen segment.
from colabdesign.mpnn import mk_mpnn_model
from Bio.PDB import PDBParser
from Bio.PDB.Polypeptide import is_aa
from Bio.SeqUtils import seq1

N_SEQ, TEMP, SEED = 5, 0.1, 0

r0      = survivors.iloc[0]
pdb0    = f"{OUT2}/{r0.variant}/{r0.design}.pdb"
trb0    = np.load(f"{OUT2}/{r0.variant}/{r0.design}.trb", allow_pickle=True)
fix_pos = sorted(int(n) for c, n in trb0["con_hal_pdb_idx"])
fix_str = ",".join(map(str, fix_pos))

print("design :", r0.design, "| length", r0.length)
print("fixed  :", len(fix_pos), "positions ->", fix_str[:60], "...")

# what the backbone already carries at those positions (IL-2's own residues)
st  = PDBParser(QUIET=True).get_structure("d", pdb0)[0]["A"]
native_at_fixed = "".join(seq1(st[p].get_resname()) for p in fix_pos)
print("motif residues in backbone:", native_at_fixed)

mpnn = mk_mpnn_model()
mpnn.set_seed(SEED)
mpnn.prep_inputs(pdb_filename=pdb0, chain="A", fix_pos=fix_str, rm_aa="C")
out = mpnn.sample_parallel(batch=N_SEQ, temperature=TEMP)

print("\nreturned keys:", list(out.keys()))
for k, v in out.items():
    print(f"  {k:10s} {type(v).__name__} "
          f"{getattr(v, 'shape', len(v) if hasattr(v, '__len__') else '')}")

seqs = out["seq"]
print("\nfirst sequence:")
print(" ", seqs[0])
print("\ncheck - motif positions preserved?")
got = "".join(seqs[0][p - 1] for p in fix_pos)
print("  expected:", native_at_fixed)
print("  got     :", got)
print("  match   :", got == native_at_fixed)
print("\nno cysteine:", "C" not in seqs[0])

design : pad0_len100_0 | length 96
fixed  : 36 positions -> 8,9,10,11,12,13,14,15,16,17,18,19,42,43,44,45,46,47,48,49,50 ...
motif residues in backbone: LQLEHLLLDLQMDLISNINVIVLENRWITFCQSIIS

returned keys: ['S', 'decoding_order', 'logits', 'seq', 'score', 'seqid']
  S          ndarray (5, 96, 21)
  decoding_order ndarray (5, 96, 1)
  logits     ndarray (5, 96, 21)
  seq        ndarray (5,)
  score      ndarray (5,)
  seqid      ndarray (5,)

first sequence:
  SEKEEIILQLEHLLLDLQMYLKMQEEAEKNPLTPEQKKLIEDLISNINVIVLELLEKLNLPEFKELLESLKKGLTKEELNRWITFCQSIISELKSE

check - motif positions preserved?
  expected: LQLEHLLLDLQMDLISNINVIVLENRWITFCQSIIS
  got     : LQLEHLLLDLQMDLISNINVIVLENRWITFCQSIIS
  match   : True

no cysteine: False


In [ ]:
# Confirm that set_seed makes sampling deterministic, so the 290
# sequences are regenerable from the recorded parameters. This is the
# reproducibility guarantee Stage 2 lacks
mpnn.set_seed(SEED)
mpnn.prep_inputs(pdb_filename=pdb0, chain="A", fix_pos=fix_str, rm_aa="C")
a = mpnn.sample_parallel(batch=N_SEQ, temperature=TEMP)["seq"]

mpnn.set_seed(SEED)
mpnn.prep_inputs(pdb_filename=pdb0, chain="A", fix_pos=fix_str, rm_aa="C")
b = mpnn.sample_parallel(batch=N_SEQ, temperature=TEMP)["seq"]

print("deterministic:", list(a) == list(b))

deterministic: True


In [ ]:
# Diagnose the surviving cysteine: C125 buries zero area against any
# receptor chain and is not a motif contact. It is a passenger inside
# the A119-130 segment, carried along because the segment is contiguous
# rather than because it contributes to binding
s1 = pd.read_csv(f"{PROJ}/stage1_motif/il2_interface_dsasa.csv")
row = s1[s1.resnum == 125]
print(row[["resnum", "resname", "d_beta", "d_gc", "d_alpha",
           "beta_contact", "gc_contact", "motif"]].to_string(index=False))

seg3 = s1[(s1.resnum >= 119) & (s1.resnum <= 130)]
print("\nsegment A119-130 — which are real contacts:")
print(seg3[["resnum", "resname", "d_beta", "d_gc", "motif"]].to_string(index=False))

 resnum resname  d_beta  d_gc  d_alpha  beta_contact  gc_contact  motif
    125     CYS     0.0   0.0      0.0         False       False  False

segment A119-130 — which are real contacts:
 resnum resname  d_beta  d_gc  motif
    119     ASN     0.0  28.4   True
    120     ARG     0.0   4.4  False
    121     TRP     0.0   0.0  False
    122     ILE     0.0   4.9  False
    123     THR     0.0  59.5   True
    124     PHE     0.0   0.0  False
    125     CYS     0.0   0.0  False
    126     GLN     0.0  81.8   True
    127     SER     0.0  30.5   True
    128     ILE     0.0   0.0  False
    129     ILE     0.0  39.5   True
    130     SER     0.0  55.4   True


In [ ]:
# Build the two fixing schemes. fixContacts intersects the grafted
# positions with the measured contact set, which drops C125 and 14 other
# passengers: 36 frozen positions become 21
s1       = pd.read_csv(f"{PROJ}/stage1_motif/il2_interface_dsasa.csv")
CONTACTS = set(s1.loc[s1.motif, "resnum"].astype(int))
print(f"{len(CONTACTS)} contact residues overall")

def fixed_positions(trb, contacts_only):
    pairs = [(int(r), int(h)) for (rc, r), (hc, h) in
             zip(trb["con_ref_pdb_idx"], trb["con_hal_pdb_idx"])]
    keep = [h for r, h in pairs if (not contacts_only) or r in CONTACTS]
    return ",".join(map(str, sorted(keep))), len(keep)

# sanity check on one design
for mode in (False, True):
    s, n = fixed_positions(trb0, mode)
    print(f"{'contacts only' if mode else 'all segment':15s} -> {n} fixed")

24 contact residues overall
all segment     -> 36 fixed
contacts only   -> 21 fixed


In [ ]:
# Full two-arm run over all 29 backbones. The cysteine count per arm is
# recorded because it is the direct consequence of the fixing scheme,
# not an incidental property of the sequences
N_SEQ, TEMP, SEED = 5, 0.1, 0
ARMS = {"fixAll": False, "fixContacts": True}

mpnn = mk_mpnn_model()
rows = []

for arm, contacts_only in ARMS.items():
    print(f"\n=== {arm} ===")
    for _, r in survivors.iterrows():
        pdb = f"{OUT2}/{r.variant}/{r.design}.pdb"
        trb = np.load(f"{OUT2}/{r.variant}/{r.design}.trb", allow_pickle=True)
        fix_str, n_fixed = fixed_positions(trb, contacts_only)

        mpnn.set_seed(SEED)
        mpnn.prep_inputs(pdb_filename=pdb, chain="A",
                         fix_pos=fix_str, rm_aa="C")
        out = mpnn.sample_parallel(batch=N_SEQ, temperature=TEMP)

        for i, (seq, sc) in enumerate(zip(out["seq"], out["score"])):
            rows.append({
                "arm": arm, "variant": r.variant, "backbone": r.design,
                "seq_id": f"{r.design}_{arm}_{i}",
                "n_fixed": n_fixed,
                "mpnn_score": round(float(sc), 4),
                "length": len(seq),
                "n_cys": seq.count("C"),
                "sequence": seq,
            })
    print(f"  {len(survivors)} backbones x {N_SEQ} sequences")

df3 = pd.DataFrame(rows)
df3.to_csv(f"{OUT3}/sequences.tsv", sep="\t", index=False)

print(f"\ntotal sequences: {len(df3)}")
print("\nMPNN score by arm (lower = more confident):")
print(df3.groupby("arm").mpnn_score.describe().round(3).to_string())
print("\ncysteines remaining:")
print(df3.groupby("arm").n_cys.sum().to_string())

with open(f"{OUT3}/params.json", "w") as fh:
    json.dump({"n_seq": N_SEQ, "temperature": TEMP, "seed": SEED,
               "rm_aa": "C", "arms": {k: ("contacts_only" if v else "all_segment")
                                      for k, v in ARMS.items()},
               "n_contacts": len(CONTACTS)}, fh, indent=2)


=== fixAll ===
  29 backbones x 5 sequences

=== fixContacts ===
  29 backbones x 5 sequences

total sequences: 290

MPNN score by arm (lower = more confident):
             count   mean    std    min    25%    50%    75%    max
arm                                                                
fixAll       145.0  1.291  0.069  1.066  1.254  1.290  1.330  1.485
fixContacts  145.0  1.196  0.062  1.039  1.160  1.204  1.236  1.330

cysteines remaining:
arm
fixAll         145
fixContacts      0


In [ ]:
# What ProteinMPNN chooses at the C125 position once it is free to
# choose. Testing, whether the model is behaving sensibly,
# since the position's environment is known independently
ref2hal = {int(r): int(h) for (rc, r), (hc, h) in
           zip(trb0["con_ref_pdb_idx"], trb0["con_hal_pdb_idx"])}
print("ref 125 -> design position", ref2hal.get(125))

from collections import Counter
subs = Counter()
for _, r in survivors.iterrows():
    trb = np.load(f"{OUT2}/{r.variant}/{r.design}.trb", allow_pickle=True)
    m = {int(a): int(b) for (ac, a), (bc, b) in
         zip(trb["con_ref_pdb_idx"], trb["con_hal_pdb_idx"])}
    pos = m.get(125)
    if pos is None:
        continue
    for s in df3[(df3.arm == "fixContacts") & (df3.backbone == r.design)].sequence:
        subs[s[pos - 1]] += 1

print("\nwhat replaced C125:")
for aa, n in subs.most_common():
    print(f"  {aa}: {n}")

ref 125 -> design position 86

what replaced C125:
  L: 108
  M: 22
  V: 10
  F: 2
  Q: 1
  H: 1
  T: 1


#### Why the cysteine filter worked in one arm and not the other?

rm_aa="C" applies only to positions ProteinMPNN is designing. Frozen positions retain whatever the backbone already carries, which at grafted positions is IL-2's own residue.

IL-2 carries a free cysteine at position 125, inside the A119-130 segment. Under fixAll that position is frozen, so the exclusion has no effect and every one of the 145 sequences retains exactly one cysteine. Under fixContacts the position is not a measured contact, so it is released, the exclusion applies, and the count falls to zero.

The difference is therefore not a property of the sequences but a direct and predictable consequence of the fixing scheme. Therefore, it demonstrates that freezing a contiguous segment imports whatever happens to lie between the contacts. C125 buries no area against any receptor chain. It was carried for 145 sequences purely because it is adjacent to residues that do. A free cysteine is a known liability: it drives disulfide scrambling and aggregation.

In [ ]:
# Independent structural check on that choice: CA neighbour counts
# within 10 A, as a burial proxy, for the C125 position in native IL-2
# and at the corresponding position across all designs
from Bio.PDB import PDBParser
from Bio.PDB.Polypeptide import is_aa
import numpy as np

def neighbour_counts(chain, radius=10.0):
    res = [r for r in chain if is_aa(r, standard=True) and "CA" in r]
    xyz = np.array([r["CA"].get_coord() for r in res])
    d   = np.linalg.norm(xyz[:, None] - xyz[None, :], axis=-1)
    return {r.id[1]: int((d[i] < radius).sum()) - 1 for i, r in enumerate(res)}

# native IL-2
nat = PDBParser(QUIET=True).get_structure("n", f"{PROJ}/2B5I.pdb")[0][CHAIN["IL2"]]
nc_nat = neighbour_counts(nat)
vals = np.array(list(nc_nat.values()))
print(f"native IL-2, residue 125: {nc_nat[125]} neighbours "
      f"({100*(vals < nc_nat[125]).mean():.0f}th percentile)")

# the same position across your designs
pcts = []
for _, r in survivors.iterrows():
    trb = np.load(f"{OUT2}/{r.variant}/{r.design}.trb", allow_pickle=True)
    m = {int(a): int(b) for (ac, a), (bc, b) in
         zip(trb["con_ref_pdb_idx"], trb["con_hal_pdb_idx"])}
    pos = m.get(125)
    if pos is None:
        continue
    ch = PDBParser(QUIET=True).get_structure(
        "d", f"{OUT2}/{r.variant}/{r.design}.pdb")[0]["A"]
    nc = neighbour_counts(ch)
    v  = np.array(list(nc.values()))
    pcts.append(100 * (v < nc[pos]).mean())

print(f"designs, same position : median {np.median(pcts):.0f}th percentile "
      f"(range {min(pcts):.0f}-{max(pcts):.0f})")

# and what native IL-2's own free SASA says
s1 = pd.read_csv(f"{PROJ}/stage1_motif/il2_interface_dsasa.csv")
print("\nnative IL-2 residue 125 free SASA:",
      s1.loc[s1.resnum == 125, "sasa_free"].values)

native IL-2, residue 125: 25 neighbours (98th percentile)
designs, same position : median 95th percentile (range 86-99)

native IL-2 residue 125 free SASA: [0.]


#### Benchmark comparison

An MPNN score of 1.196 is uninterpretable in isolation. Below is a reference scale by applying the identical protocol to native IL-2, to Neo-2/15, and to native IL-2, carrying the same frozen positions as the designs.

In [ ]:
# Redesign two reference scaffolds unconstrained under the same protocol,
# giving an upper and lower anchor: a natural cytokine and a published
# de novo design. Sequence recovery against the native sequence is
# recorded as a check that the model is operating normally
import urllib.request
from Bio.PDB import PDBParser, PDBIO, Select
from Bio.PDB.Polypeptide import is_aa
from Bio.SeqUtils import seq1

BENCH = f"{OUT3}/benchmark"
os.makedirs(BENCH, exist_ok=True)

# --- Neo-2/15 ---
neo_raw = f"{BENCH}/6DG6.pdb"
if not os.path.exists(neo_raw):
    urllib.request.urlretrieve(
        "https://files.rcsb.org/download/6DG6.pdb", neo_raw)

class One(Select):
    def __init__(self, cid): self.cid = cid
    def accept_chain(self, c):  return c.id == self.cid
    def accept_residue(self, r): return is_aa(r, standard=True)
    def accept_atom(self, a):    return a.element != "H"

io = PDBIO()
refs = {}
for name, src, cid in [("IL2", f"{PROJ}/2B5I.pdb", CHAIN["IL2"]),
                       ("Neo-2/15", neo_raw, "A")]:
    st = PDBParser(QUIET=True).get_structure(name, src)[0]
    io.set_structure(st)
    path = f"{BENCH}/{name.replace('/', '')}.pdb"
    io.save(path, One(cid))
    seq = "".join(seq1(r.get_resname()) for r in st[cid]
                  if is_aa(r, standard=True))
    refs[name] = (path, seq)
    print(f"{name:10s} {len(seq)} residues  {seq[:40]}...")

# --- redesign each with nothing fixed, same protocol as Stage 3 ---
bench_rows = []
for name, (path, native_seq) in refs.items():
    mpnn.set_seed(SEED)
    mpnn.prep_inputs(pdb_filename=path, chain="A", rm_aa="C")
    out = mpnn.sample_parallel(batch=10, temperature=TEMP)
    for s, sc in zip(out["seq"], out["score"]):
        rec = sum(a == b for a, b in zip(s, native_seq)) / len(native_seq)
        bench_rows.append({"target": name, "score": float(sc),
                           "recovery": rec, "sequence": s})

bench = pd.DataFrame(bench_rows)
print("\n--- benchmark ---")
print(bench.groupby("target")[["score", "recovery"]]
           .agg(["mean", "std"]).round(3).to_string())

print("\n--- your designs, for comparison ---")
print(df3.groupby("arm").mpnn_score.agg(["mean", "std"]).round(3).to_string())

bench.to_csv(f"{BENCH}/benchmark.tsv", sep="\t", index=False)

IL2        120 residues  STKKTQLQLEHLLLDLQMILNGINNYKNPKLTRMLTFKFY...
Neo-2/15   100 residues  PKKKIQLHAEHALYDALMILNIVKTNSPPAEEKLEDYAFN...

--- benchmark ---
          score        recovery       
           mean    std     mean    std
target                                
IL2       1.069  0.030    0.365  0.023
Neo-2/15  0.896  0.055    0.323  0.031

--- your designs, for comparison ---
              mean    std
arm                      
fixAll       1.291  0.069
fixContacts  1.196  0.062


In [ ]:
# Control: freeze the same 21 and 36 positions on native IL-2 itself.
# Without it, the designs' elevated score could reflect either poor
# backbones or the cost of the graft --> both are indistinguishable.
s1_       = pd.read_csv(f"{PROJ}/stage1_motif/il2_interface_dsasa.csv")
contacts  = sorted(s1_.loc[s1_.motif, "resnum"].astype(int))
segments  = [r for seg in [(12, 23), (84, 95), (119, 130)]
             for r in range(seg[0], seg[1] + 1)]

fair = []
for label, keep in [("IL2_free", None),
                    ("IL2_fix21", [r for r in contacts if r in segments]),
                    ("IL2_fix36", segments)]:
    mpnn.set_seed(SEED)
    mpnn.prep_inputs(pdb_filename=refs["IL2"][0], chain="A", rm_aa="C",
                     fix_pos=",".join(map(str, keep)) if keep else None)
    out = mpnn.sample_parallel(batch=10, temperature=TEMP)
    for s, sc in zip(out["seq"], out["score"]):
        fair.append({"target": label,
                     "n_fixed": len(keep) if keep else 0,
                     "score": float(sc)})

fair = pd.DataFrame(fair)
print(fair.groupby(["target", "n_fixed"]).score.agg(["mean", "std"]).round(3))
print("\nyour designs:")
print(df3.groupby("arm").mpnn_score.agg(["mean", "std"]).round(3).to_string())
fair.to_csv(f"{BENCH}/fair_comparison.tsv", sep="\t", index=False)

                    mean    std
target    n_fixed              
IL2_fix21 21       1.031  0.020
IL2_fix36 36       1.053  0.021
IL2_free  0        1.069  0.030

your designs:
              mean    std
arm                      
fixAll       1.291  0.069
fixContacts  1.196  0.062


In [ ]:
# Redesign the 29 backbones with nothing fixed, completing the ladder.
# Free-vs-free comparison isolates backbone designability.
# Fixed-vs-free difference within each scaffold isolates graft cost.
free_rows = []
for _, r in survivors.iterrows():
    pdb = f"{OUT2}/{r.variant}/{r.design}.pdb"
    mpnn.set_seed(SEED)
    mpnn.prep_inputs(pdb_filename=pdb, chain="A", rm_aa="C")   # nothing fixed
    out = mpnn.sample_parallel(batch=5, temperature=TEMP)
    for s, sc in zip(out["seq"], out["score"]):
        free_rows.append({"backbone": r.design, "score": float(sc)})

free_df = pd.DataFrame(free_rows)
free_df.to_csv(f"{BENCH}/designs_unconstrained.tsv", sep="\t", index=False)

print("--- full ladder ---")
rows = [
    ("Neo-2/15, free",            0.896),
    ("IL-2, 21 fixed",            1.031),
    ("IL-2, 36 fixed",            1.053),
    ("IL-2, free",                1.069),
    ("your backbones, free",      free_df.score.mean()),
    ("your backbones, 21 fixed",  df3[df3.arm == "fixContacts"].mpnn_score.mean()),
    ("your backbones, 36 fixed",  df3[df3.arm == "fixAll"].mpnn_score.mean()),
]
for label, v in rows:
    print(f"  {label:28s} {v:.3f}")

bb   = free_df.score.mean()
cost = df3[df3.arm == "fixContacts"].mpnn_score.mean() - bb
print(f"\nbackbone designability gap vs native IL-2 : {bb - 1.069:+.3f}")
print(f"cost of the frozen motif on your backbones: {cost:+.3f}")

--- full ladder ---
  Neo-2/15, free               0.896
  IL-2, 21 fixed               1.031
  IL-2, 36 fixed               1.053
  IL-2, free                   1.069
  your backbones, free         1.078
  your backbones, 21 fixed     1.196
  your backbones, 36 fixed     1.291

backbone designability gap vs native IL-2 : +0.009
cost of the frozen motif on your backbones: +0.117


In [ ]:
# Persist the ladder as a table, since it is the quantitative result of
# this stage and is referenced by the attrition figure.
ladder = pd.DataFrame([
    ("Neo-2/15",        "free",     0.896),
    ("IL-2",            "free",     1.069),
    ("IL-2",            "21 fixed", 1.031),
    ("IL-2",            "36 fixed", 1.053),
    ("designs",         "free",     round(free_df.score.mean(), 3)),
    ("designs",         "21 fixed", 1.196),
    ("designs",         "36 fixed", 1.291),
], columns=["target", "condition", "mpnn_score"])
ladder.to_csv(f"{BENCH}/ladder.tsv", sep="\t", index=False)
print(ladder.to_string(index=False))

  target condition  mpnn_score
Neo-2/15      free       0.896
    IL-2      free       1.069
    IL-2  21 fixed       1.031
    IL-2  36 fixed       1.053
 designs      free       1.078
 designs  21 fixed       1.196
 designs  36 fixed       1.291


#### Results and conclusion
1. 290 sequences, deterministic and regenerable. 29 backbones x2 arms x5 sequences, seed 0, temperature 0.1. Re-running the sampler with the same seed reproduces the exactly the same sequences.
2. FixContacts outperforms fixAll on both measurable criteria.

| arm | frozen | MPNN score | cysteines |
|---|---|---|---|
| fixAll | 36 | 1.291 ± 0.069 | 145 |
| fixContacts | 21 | 1.196 ± 0.062 | 0 |

3. Freezing only the measured contacts gives a 0.095 better score and eliminates the free cysteine. Releasing the 15 non-contact passenger positions costs nothing in retained binding geometry, since by construction they bury no receptor surface.
4. The benchmark ladder:

| scaffold | condition | MPNN score |
|---|---|---|
| Neo-2/15 | free | 0.896 |
| IL-2, 21 frozen | frozen | 1.031 |
| IL-2, 36 frozen | frozen | 1.053 |
| IL-2 | free | 1.069 |
| designed backbones | free | 1.078 |
| designed backbones | 21 frozen | 1.196 |
| designed backbones | 36 frozen | 1.291 |

5. The generated backbones are as designable as native IL-2: redesigned unconstrained score 1.078 against native IL-2's 1.069, which is inside the between-sequence standard deviation. RFdiffusion produced backbones that ProteinMPNN finds no harder to encode than a real cytokine.

6. Neo-2/15 scores better than any natural protein in this study. At 0.896 it is markedly more designable than native IL-2, as it is expected for an idealised de novo scaffold, demonstrating that MPNN score measures idealness of sequence-structure agreement, not function.

#### Conclusion
1. Unconstrained designability matches native IL-2 to within
   0.009 under ProteinMPNN’s own model. Stage 4 shows MPNN score carries no within-arm information about foldability, so this says the backbones are as easy to encode as a real cytokine, however, not that they fold

2. fixContacts is the better arm, on evidence rather than on preference, as is scores better, demonstrates 0 cysteine liability, and shows no loss of contact geometry.

3. MPNN score is a designability metric and was not used for selection. It ranks sequence-structure agreement. Nothing here establishes folding, stability or binding.

#### Limitations and future improvements
1.  Pairwise identity within each backbone group was not computed
2.  Sequences were designed on the monomer alone.
3.  ProteinMPNN reports a median native recovery near 50% at this temperature. Native IL-2 recovers at 36.5%. Three contributing factors are identifiable: cysteine is forbidden and IL-2 has three, the chain is taken from a complex and designed in isolation, and the 2B5I chain has unresolved gaps.
4. One sampling temperature, one seed.
